In [1]:
"""
Calorimeter Hit Transformer with Energy-Weighted Attention Bias
---------------------------------------------------------------
Input : (B, N, 4)  — B batches, N hits, features [x, y, z, E]
Output: (B, d_model) — per-event embedding via [CLS] token readout

Energy-weighted attention bias:
    bias_ij = w * (E_i * E_j) / (sum_E)^2
added to the raw QK^T scores before softmax so the network
attends more strongly to energetic hit pairs by default,
while still being free to learn away from this prior.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math


# ── Helpers ────────────────────────────────────────────────────────────────────

def energy_weighted_bias(E: torch.Tensor, padding_mask: torch.Tensor | None = None) -> torch.Tensor:
    """
    Compute per-head-broadcastable energy-weight bias matrix.

    Args:
        E            : (B, N+1)  raw energies (CLS energy set to 0)
        padding_mask : (B, N+1)  bool, True = padded / ignore

    Returns:
        bias : (B, 1, N+1, N+1)  — broadcast over h heads
    """
    # Mask padded hits before summing energy
    if padding_mask is not None:
        E = E.masked_fill(padding_mask, 0.0)

    sum_E = E.sum(dim=-1, keepdim=True).clamp(min=1e-6)   # (B, 1)
    E_norm = E / sum_E                                      # (B, N+1)

    # outer product → (B, N+1, N+1)
    bias = torch.bmm(E_norm.unsqueeze(-1), E_norm.unsqueeze(-2))
    return bias.unsqueeze(1)                                # (B, 1, N+1, N+1)


# ── Core Attention ──────────────────────────────────────────────────────────────

class EnergyBiasedMHA(nn.Module):
    """
    Multi-head self-attention with an additive energy-weighted bias.

        scores_ij = (Q_i · K_j) / sqrt(d_k)  +  alpha * bias_ij

    alpha is a learnable scalar (initialised to 1.0) so the network
    can dial the physics prior up or down during training.
    """

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model  = d_model
        self.n_heads  = n_heads
        self.d_k      = d_model // n_heads

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout  = nn.Dropout(dropout)

        # Learnable scale for the energy bias
        self.alpha = nn.Parameter(torch.ones(1))

    def forward(
        self,
        x:            torch.Tensor,               # (B, S, d_model)
        energy_bias:  torch.Tensor,               # (B, 1, S, S)
        key_pad_mask: torch.Tensor | None = None, # (B, S) bool, True=ignore
    ) -> torch.Tensor:
        B, S, _ = x.shape

        # Project to Q, K, V and split heads
        qkv = self.qkv_proj(x).reshape(B, S, 3, self.n_heads, self.d_k)
        qkv = qkv.permute(2, 0, 3, 1, 4)          # (3, B, h, S, d_k)
        Q, K, V = qkv.unbind(0)                    # each (B, h, S, d_k)

        # Scaled dot-product scores
        scale  = math.sqrt(self.d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / scale  # (B, h, S, S)

        # Add energy bias (broadcast over heads)
        scores = scores + self.alpha * energy_bias              # (B, h, S, S)

        # Mask padded keys
        if key_pad_mask is not None:
            # (B, 1, 1, S) → broadcast over queries and heads
            scores = scores.masked_fill(
                key_pad_mask[:, None, None, :], float("-inf")
            )

        attn   = self.dropout(F.softmax(scores, dim=-1))
        out    = torch.matmul(attn, V)                         # (B, h, S, d_k)
        out    = out.transpose(1, 2).reshape(B, S, self.d_model)
        return self.out_proj(out)


# ── Transformer Block ───────────────────────────────────────────────────────────

class CaloTransformerBlock(nn.Module):
    """
    Pre-norm transformer block:
        x = x + MHA(LN(x))
        x = x + FFN(LN(x))
    """

    def __init__(self, d_model: int, n_heads: int, ffn_mult: int = 4, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn  = EnergyBiasedMHA(d_model, n_heads, dropout)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, ffn_mult * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_mult * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(
        self,
        x:            torch.Tensor,
        energy_bias:  torch.Tensor,
        key_pad_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        x = x + self.attn(self.norm1(x), energy_bias, key_pad_mask)
        x = x + self.ffn(self.norm2(x))
        return x


# ── Full Model ──────────────────────────────────────────────────────────────────

class CaloHitTransformer(nn.Module):
    """
    Transformer encoder for calorimeter events.

    Input hit tensor : (B, N, 4)  with features [x, y, z, E]
    Padding mask     : (B, N)     bool, True = padded hit (optional)

    Output           : (B, d_model)  event-level embedding

    Architecture:
        1. Log-scale energy   E → log(1 + E)
        2. Linear projection  (B, N+1, 4) → (B, N+1, d_model)
        3. L × CaloTransformerBlock  (with energy-weighted attention bias)
        4. Extract [CLS] token → event embedding
        5. Pluggable task head
    """

    def __init__(
        self,
        d_model:    int   = 128,
        n_heads:    int   = 8,
        n_layers:   int   = 6,
        ffn_mult:   int   = 4,
        dropout:    float = 0.1,
        n_classes:  int | None = None,   # set for classification
        n_outputs:  int | None = None,   # set for regression
        log_energy: bool  = True,        # log-scale raw E before projection
    ):
        super().__init__()
        self.log_energy = log_energy

        # [CLS] token (learnable, broadcast over batch at forward time)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, 4))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        # Input projection: 4 → d_model
        self.input_proj = nn.Linear(4, d_model)

        # Transformer layers
        self.layers = nn.ModuleList([
            CaloTransformerBlock(d_model, n_heads, ffn_mult, dropout)
            for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)

        # Task head (optional — swap freely)
        if n_classes is not None:
            self.head = nn.Sequential(
                nn.Linear(d_model, d_model // 2),
                nn.GELU(),
                nn.Linear(d_model // 2, n_classes),
            )
        elif n_outputs is not None:
            self.head = nn.Sequential(
                nn.Linear(d_model, d_model // 2),
                nn.GELU(),
                nn.Linear(d_model // 2, n_outputs),
            )
        else:
            self.head = nn.Identity()   # return raw embedding

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(
        self,
        hits:         torch.Tensor,               # (B, N, 4)  [x, y, z, E]
        padding_mask: torch.Tensor | None = None, # (B, N) bool, True=padded
    ) -> torch.Tensor:
        B, N, _ = hits.shape

        # 1. Log-scale energy channel only
        if self.log_energy:
            hits = hits.clone()
            hits[..., 3] = torch.log1p(hits[..., 3])

        # 2. Prepend [CLS] token
        cls = self.cls_token.expand(B, -1, -1)    # (B, 1, 4)
        x   = torch.cat([cls, hits], dim=1)        # (B, N+1, 4)

        # 3. Build energy tensor for bias (CLS gets E=0, no contribution)
        E_full = torch.cat([
            torch.zeros(B, 1, device=hits.device, dtype=hits.dtype),
            hits[..., 3],                          # log-scaled E
        ], dim=1)                                  # (B, N+1)

        # Extend padding mask to cover CLS (never masked)
        if padding_mask is not None:
            cls_mask = torch.zeros(B, 1, dtype=torch.bool, device=hits.device)
            pad_full = torch.cat([cls_mask, padding_mask], dim=1)  # (B, N+1)
        else:
            pad_full = None

        # 4. Compute energy bias once (reused by all layers)
        e_bias = energy_weighted_bias(E_full, pad_full)  # (B, 1, N+1, N+1)

        # 5. Linear projection
        x = self.input_proj(x)                     # (B, N+1, d_model)

        # 6. Transformer layers
        for layer in self.layers:
            x = layer(x, e_bias, pad_full)

        x = self.final_norm(x)

        # 7. CLS readout → event embedding
        event_emb = x[:, 0]                        # (B, d_model)

        return self.head(event_emb)


# ── Quick smoke-test ────────────────────────────────────────────────────────────

if __name__ == "__main__":
    torch.manual_seed(42)

    B, N = 4, 128   # 4 events, up to 128 hits each

    # Random hits: x,y,z in [-1500, 1500] mm, E in [0, 500] GeV
    hits = torch.cat([
        torch.rand(B, N, 3) * 3000 - 1500,
        torch.rand(B, N, 1) * 500,
    ], dim=-1)

    # Simulate variable-length events: last 20 hits of events 2 & 3 are padding
    padding_mask = torch.zeros(B, N, dtype=torch.bool)
    padding_mask[2, 108:] = True
    padding_mask[3,  80:] = True

    # ── Classification model (e.g. 5 particle types) ──
    clf_model = CaloHitTransformer(
        d_model=128, n_heads=8, n_layers=4,
        dropout=0.1, n_classes=5,
    )
    logits = clf_model(hits, padding_mask)
    print(f"Classification logits : {logits.shape}")  # (4, 5)

    # ── Regression model (e.g. energy + 2 angles) ──
    reg_model = CaloHitTransformer(
        d_model=128, n_heads=8, n_layers=4,
        dropout=0.1, n_outputs=3,
    )
    preds = reg_model(hits, padding_mask)
    print(f"Regression outputs    : {preds.shape}")   # (4, 3)

    # ── Embedding-only (no head) ──
    emb_model = CaloHitTransformer(d_model=128, n_heads=8, n_layers=4)
    emb = emb_model(hits, padding_mask)
    print(f"Event embeddings      : {emb.shape}")     # (4, 128)

    # Parameter count
    total = sum(p.numel() for p in clf_model.parameters())
    print(f"Parameters (clf)      : {total:,}")

Classification logits : torch.Size([4, 5])
Regression outputs    : torch.Size([4, 3])
Event embeddings      : torch.Size([4, 128])
Parameters (clf)      : 800,525


In [2]:
from torch_cluster import knn_graph

ModuleNotFoundError: No module named 'torch_cluster'

In [3]:
%pip install torch_cluster


  Using cached torch_cluster-1.6.3.tar.gz (54 kB)
  Installing build dependencies ... one
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [20 lines of output]
      Traceback (most recent call last):
        File "/global/homes/a/aneekj02/.conda/envs/colliderml-env/lib/python3.11/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 389, in <module>
          main()
        File "/global/homes/a/aneekj02/.conda/envs/colliderml-env/lib/python3.11/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "/global/homes/a/aneekj02/.conda/envs/colliderml-env/lib/python3.11/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 143, in get_requires_for

In [4]:
from torch_geometric.nn import GravNetConv, global_mean_pool, global_add_pool


/global/homes/a/aneekj02/.conda/envs/colliderml-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
